# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Goal:** build two retrieval pipelines over the *same* chunked document and compare them.

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder retrieves Top 10 by cosine similarity | *same* Top 10 from Pipeline 1 |
| Stage 2 | – | Cross-encoder scores all 10 (query, chunk) pairs, re-sorts descending |
| Context for the LLM | Top 3 by cosine | Top 3 by cross-encoder score |
| Output | Answer 1 | Answer 2 (identical prompt template) |

**Document:** *Regulation (EU) 2016/679 – General Data Protection Regulation (GDPR)*, official EUR-Lex PDF (88 pages).
It is a good stress test for retrieval because:
- 173 recitals (non-binding explanations) paraphrase the 99 binding articles, so there are many near-duplicate distractors;
- rules are full of exceptions (*"unless…"*, *"shall not apply to…"*), negations and role-specific obligations (controller vs. processor vs. representative);
- many provisions refer to each other only by number (*"Articles 25 to 39"*).

**Models**
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2` (384-d embeddings)
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Vector store: FAISS `IndexFlatIP` over L2-normalised vectors (inner product = cosine similarity)
- LLM: Google Gemini (`gemini-2.5-flash`, temperature 0)

**Notebook layout**
1. Setup
2. Loading & cleaning the PDF
3. Chunking
4. Pipeline 1 – baseline retrieval
5. Pipeline 2 – cross-encoder re-ranking
6. Answer generation (shared prompt)
7. Failure mode analysis
8. Summary metrics
9. Latency profiling
10. Scaling experiment

## 1. Setup

All tunable settings are kept in one place. The Gemini API key is read from a local `.env` file
(`GEMINI_API_KEY=...`), which is git-ignored – see `.env.example`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os, re, json, time, random, hashlib, platform
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PyMuPDF, for PDF text + font information
import faiss                        # vector index
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv
from IPython.display import display, Markdown
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()          # hide "Loading weights" bars
pd.set_option("display.max_colwidth", 120)

# ---- configuration ----
PDF_PATH        = Path("data/GDPR_EU_2016_679.pdf")
RESULTS_DIR     = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CHUNK_MAX_WORDS = 150   # ~200 word-piece tokens, stays under MiniLM's 256-token limit
CHUNK_OVERLAP   = 30    # words carried over between consecutive chunks of the same article
TOP_K           = 10    # candidates retrieved by the bi-encoder
TOP_N           = 3     # chunks given to the LLM

load_dotenv()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# fixed seeds so that everything except API latency is repeatable
random.seed(0); np.random.seed(0); torch.manual_seed(0)

print("Python  :", platform.python_version())
print("Machine :", platform.platform())
print("CPU     :", platform.processor() or platform.machine(), "|", os.cpu_count(), "logical cores")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")
print("Torch threads:", torch.get_num_threads())

Python  : 3.10.13
Machine : Linux-7.0.0-38-generic-x86_64-with-glibc2.39
CPU     : x86_64 | 8 logical cores
GPU     : none (CPU only)
Torch threads: 4


## 2. Loading & cleaning the PDF

Plain `get_text()` mixes three kinds of noise into the regulation text:
- the running footer on every page (`4.5.2016`, `L 119/12`, `Official Journal of the European Union`, `EN`),
- footnotes (`(1) OJ C 229, 31.7.2012, p. 90.`),
- chapter / section headings.

Instead of guessing with regexes alone, I use the **font information** PyMuPDF exposes:
body text is set in 9.6 pt, footnotes and the `EN` tag in 8.5 pt, article headings (`Article 17`) in italic and
article titles in bold. Each line is kept together with its page number and bold/italic flags.

In [2]:
FOOTER_RE  = re.compile(r"^(\d{1,2}\.\d{1,2}\.\d{4}|L \d+/\d+|Official Journal of the European Union)$")
BODY_SIZE  = 9.0          # anything smaller than this is a footnote / page tag
BOLD, ITALIC = 16, 2      # PyMuPDF span flag bits


def load_lines(path):
    # Return every body-text line as {page, text, bold, italic}.
    lines = []
    for page_no, page in enumerate(pymupdf.open(path), start=1):
        for block in page.get_text("dict")["blocks"]:
            for line in block.get("lines", []):
                spans = [s for s in line["spans"] if s["size"] >= BODY_SIZE]   # drop footnote-sized spans
                text = "".join(s["text"] for s in spans).strip()
                if not text or FOOTER_RE.match(text):                          # drop the running footer
                    continue
                flags = spans[0]["flags"]
                lines.append({"page": page_no, "text": text,
                              "bold": bool(flags & BOLD), "italic": bool(flags & ITALIC)})
    return lines


lines = load_lines(PDF_PATH)
print(f"{pymupdf.open(PDF_PATH).page_count} pages -> {len(lines)} body-text lines")
pd.DataFrame(lines[:8])

88 pages -> 4124 body-text lines


,page,text,bold,italic
0,1,I,False,False
1,1,(Legislative acts),False,True
2,1,REGULATIONS,False,False
3,1,REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL,True,False
4,1,of 27 April 2016,True,False
5,1,on the protection of natural persons with regard to the processing of personal data and on the free,True,False
6,1,"movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation)",True,False
7,1,"Having regard to the Treaty on the Functioning of the European Union, and in particular Article 16 thereof,",False,False


### 2.1 Splitting into recitals and articles

The regulation has two parts:
- **Recitals** `(1)` … `(173)` – the "whereas" explanations (pages 1–31);
- **Articles** `Article 1` … `Article 99` – the binding rules (pages 32–88).

I group lines into one *unit* per recital / article and keep the article title as metadata
(e.g. `Article 33 - Notification of a personal data breach to the supervisory authority`).
A recital number only starts a new recital if it is the next number in sequence, so inline
references like "(4)" do not break the text. Words hyphenated across line breaks are re-joined.

In [3]:
ARTICLE_RE = re.compile(r"^Article (\d+)$")
RECITAL_RE = re.compile(r"^\((\d+)\)\s*(.*)$")


def to_units(lines):
    units, cur = [], None
    in_articles, title_open, next_recital = False, False, 1

    def flush():
        if cur and cur["text"].strip():
            units.append(cur)

    for ln in lines:
        t = ln["text"]
        # the first italic 'Article 1' heading marks the end of the recitals
        if not in_articles and ARTICLE_RE.match(t) and ln["italic"]:
            in_articles = True

        if not in_articles:
            if ln["bold"]:
                continue                                    # title block / headings
            m = RECITAL_RE.match(t)
            if m and int(m.group(1)) == next_recital:
                flush()
                cur = {"section": f"Recital ({next_recital})", "page": ln["page"], "text": m.group(2)}
                next_recital += 1
            elif cur is not None:
                cur["text"] += " " + t
            continue

        m = ARTICLE_RE.match(t)
        if m and ln["italic"]:                              # new article heading
            flush()
            cur = {"section": f"Article {m.group(1)}", "page": ln["page"], "text": ""}
            title_open = True
            continue
        if ln["bold"]:
            if cur is not None and title_open:              # article title, can wrap over 2-3 lines
                sep = " - " if " - " not in cur["section"] else " "
                cur["section"] += sep + t
            continue                                        # chapter / section headings are dropped
        title_open = False
        if cur is not None:
            cur["text"] += " " + t
    flush()

    for u in units:
        txt = re.sub(r"(\w)- (\w)", r"\1\2", u["text"])     # re-join hyphenated words
        u["text"] = re.sub(r"\s+", " ", txt).strip()
    return units


units = to_units(lines)
n_rec = sum(u["section"].startswith("Recital") for u in units)
n_art = sum(u["section"].startswith("Article") for u in units)
print(f"{n_rec} recitals, {n_art} articles")
assert n_rec == 173 and n_art == 99, "structure parsing went wrong"
pd.DataFrame(units)[["section", "page", "text"]].iloc[[0, 172, 173, 205]]

173 recitals, 99 articles


,section,page,text
0,Recital (1),1,The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1...
172,Recital (173),31,This Regulation should apply to all matters concerning the protection of fundamental rights and freedoms vis-àvis th...
173,Article 1 - Subject-matter and objectives,32,1. This Regulation lays down rules relating to the protection of natural persons with regard to the processing of pe...
205,Article 33 - Notification of a personal data breach to the supervisory authority,52,"1. In the case of a personal data breach, the controller shall without undue delay and, where feasible, not later th..."


## 3. Chunking

Each recital/article is split into chunks of at most **150 words** with a **30-word overlap**, cutting on
sentence (or `;` / `:`) boundaries so that numbered paragraphs stay intact. Chunks never cross an article
boundary, so every chunk carries a clean `section` label and page number – useful for checking the LLM's
citations later.

150 words is chosen to fit the bi-encoder: `all-MiniLM-L6-v2` truncates input at 256 word-piece tokens, and
anything past that limit would silently be ignored during embedding. The check below confirms no chunk is
truncated.

This list of chunks is built **once** and is used by both pipelines.

In [4]:
SENT_SPLIT_RE = re.compile(r"(?<=[.;:])\s+(?=[A-Z(0-9])")


def chunk_units(units, max_words=CHUNK_MAX_WORDS, overlap=CHUNK_OVERLAP):
    chunks = []
    for u in units:
        # sentence-like pieces; very long sentences are hard-split with overlap
        pieces = []
        for sent in SENT_SPLIT_RE.split(u["text"]):
            words = sent.split()
            while len(words) > max_words:
                pieces.append(" ".join(words[:max_words]))
                words = words[max_words - overlap:]
            if words:
                pieces.append(" ".join(words))

        # greedily pack pieces into chunks, carrying the last `overlap` words forward
        buf = []
        for piece in pieces:
            if buf and len(" ".join(buf + [piece]).split()) > max_words:
                chunks.append({"section": u["section"], "page": u["page"], "text": " ".join(buf)})
                buf = [" ".join(" ".join(buf).split()[-overlap:])]
            buf.append(piece)
        if buf:
            chunks.append({"section": u["section"], "page": u["page"], "text": " ".join(buf)})

    for i, c in enumerate(chunks):
        c["chunk_id"] = i
    return chunks


chunks = chunk_units(units)          # <- the single shared corpus for both pipelines
chunk_texts = [c["text"] for c in chunks]
chunks_df = pd.DataFrame(chunks).set_index("chunk_id")

word_counts = chunks_df["text"].str.split().str.len()
print(f"{len(chunks)} chunks "
      f"({chunks_df.section.str.startswith('Recital').sum()} from recitals, "
      f"{chunks_df.section.str.startswith('Article').sum()} from articles)")
word_counts.describe().round(1).to_frame("words per chunk").T

565 chunks (258 from recitals, 307 from articles)


,count,mean,std,min,25%,50%,75%,max
words per chunk,565.0,109.5,32.9,22.0,85.0,119.0,137.0,164.0


In [5]:
# Make sure no chunk exceeds the bi-encoder's 256 token window (otherwise its tail is never embedded)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BI_ENCODER_NAME)
token_counts = pd.Series([len(tok(t)["input_ids"]) for t in chunk_texts])
print("tokens per chunk: mean %.0f | max %d | chunks over 256 tokens: %d"
      % (token_counts.mean(), token_counts.max(), (token_counts > 256).sum()))

# a few sample chunks
chunks_df.loc[[0, 323, 366]]

tokens per chunk: mean 128 | max 196 | chunks over 256 tokens: 0


,section,page,text
chunk_id,,,
0,Recital (1),1,The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1...
323,Article 17 - Right to erasure (‘right to be forgotten’),43,1. The data subject shall have the right to obtain from the controller the erasure of personal data concerning him o...
366,Article 33 - Notification of a personal data breach to the supervisory authority,52,"1. In the case of a personal data breach, the controller shall without undue delay and, where feasible, not later th..."


## 4. Pipeline 1 – Baseline retrieval (bi-encoder)

**(a) Indexing.** Every chunk is encoded *independently* by the bi-encoder into a 384-d vector.
Vectors are L2-normalised, so the inner product computed by `IndexFlatIP` equals cosine similarity.
`IndexFlatIP` is an exact (brute-force) index, which is fine for ~560 vectors.

This indexing work is done once, offline; at query time only the query has to be encoded.

In [6]:
t0 = time.perf_counter()
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")
bi_load_s = time.perf_counter() - t0

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
index_build_s = time.perf_counter() - t0

print(f"bi-encoder loaded in {bi_load_s:.2f}s")
print(f"embedded + indexed {index.ntotal} chunks (dim={chunk_emb.shape[1]}) in {index_build_s:.2f}s")
print("vectors normalised:", np.allclose(np.linalg.norm(chunk_emb, axis=1), 1.0, atol=1e-4))

bi-encoder loaded in 10.25s
embedded + indexed 565 chunks (dim=384) in 30.28s
vectors normalised: True


**(b) Retrieval.** The query is embedded with the same model and the 10 nearest chunks by cosine similarity are returned.

**(c) Selection.** The baseline simply takes the first 3 of those 10 – no re-ordering.

In [7]:
def retrieve_top_k(query, k=TOP_K):
    # Bi-encoder retrieval: returns k candidates ordered by cosine similarity.
    q_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    return [{"chunk_id": int(cid), "bi_rank": rank, "bi_score": float(s)}
            for rank, (cid, s) in enumerate(zip(ids[0], scores[0]), start=1)]


def baseline_top_n(candidates, n=TOP_N):
    # Pipeline 1: take the Top-n straight from the bi-encoder order.
    return candidates[:n]


demo_q = "How quickly must a personal data breach be notified to the supervisory authority?"
demo_top10 = retrieve_top_k(demo_q)
pd.DataFrame(demo_top10).assign(section=lambda d: d.chunk_id.map(chunks_df.section),
                                text=lambda d: d.chunk_id.map(chunks_df.text).str[:90])

,chunk_id,bi_rank,bi_score,section,text
0,366,1,0.776661,Article 33 - Notification of a personal data breach to the supervisory authority,"1. In the case of a personal data breach, the controller shall without undue delay and, wh"
1,134,2,0.775250,Recital (87),It should be ascertained whether all appropriate technological protection and organisation
2,132,3,0.735003,Recital (85),"unauthorised reversal of pseudonymisation, damage to reputation, loss of confidentiality o"
3,371,4,0.659280,Article 34 - Communication of a personal data breach to the data subject,"would involve disproportionate effort. In such a case, there shall instead be a public com"
4,133,5,0.637348,Recital (86),"The controller should communicate to the data subject a personal data breach, without undu"
5,135,6,0.593405,Recital (88),In setting detailed rules concerning the format and procedures applicable to the notificat
6,369,7,0.591959,Article 34 - Communication of a personal data breach to the data subject,1. When the personal data breach is likely to result in a high risk to the rights and free
7,368,8,0.573150,Article 33 - Notification of a personal data breach to the supervisory authority,phases without undue further delay. 5. The controller shall document any personal data bre
8,367,9,0.573057,Article 33 - Notification of a personal data breach to the supervisory authority,"the nature of the personal data breach including where possible, the categories and approx"
9,510,10,0.541692,Article 70 - Tasks of the Board,"issue guidelines, recommendations and best practices in accordance with point (e) of this"


## 5. Pipeline 2 – Cross-encoder re-ranking

**(a)** Start from exactly the same Top-10 list that Pipeline 1 retrieved (no second retrieval).
**(b)** Feed each `(query, chunk)` pair through the cross-encoder. Unlike the bi-encoder, it reads the query and the
chunk *together* in one transformer pass, so every query token can attend to every chunk token; it outputs a single
relevance logit (higher = more relevant).
**(c)** Sort the 10 candidates by that score, descending, and keep the Top 3.

In [8]:
t0 = time.perf_counter()
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")
ce_load_s = time.perf_counter() - t0
print(f"cross-encoder loaded in {ce_load_s:.2f}s, max_length={cross_encoder.max_length}")


def rerank(query, candidates):
    # Score every (query, chunk) pair jointly and re-sort the candidates by cross-encoder score.
    pairs = [(query, chunk_texts[c["chunk_id"]]) for c in candidates]
    scores = cross_encoder.predict(pairs, batch_size=len(pairs), show_progress_bar=False)
    reranked = [{**c, "ce_score": float(s)} for c, s in zip(candidates, scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)
    for rank, c in enumerate(reranked, start=1):
        c["ce_rank"] = rank
    return reranked


def reranked_top_n(reranked, n=TOP_N):
    # Pipeline 2: Top-n after cross-encoder re-sorting.
    return reranked[:n]


demo_reranked = rerank(demo_q, demo_top10)
(pd.DataFrame(demo_reranked)
   .assign(section=lambda d: d.chunk_id.map(chunks_df.section),
           text=lambda d: d.chunk_id.map(chunks_df.text).str[:80])
   [["ce_rank", "bi_rank", "chunk_id", "ce_score", "bi_score", "section", "text"]])

cross-encoder loaded in 10.18s, max_length=512


,ce_rank,bi_rank,chunk_id,ce_score,bi_score,section,text
0,1,1,366,8.232859,0.776661,Article 33 - Notification of a personal data breach to the supervisory authority,"1. In the case of a personal data breach, the controller shall without undue del"
1,2,3,132,7.903715,0.735003,Recital (85),"unauthorised reversal of pseudonymisation, damage to reputation, loss of confide"
2,3,2,134,6.309049,0.775250,Recital (87),It should be ascertained whether all appropriate technological protection and or
3,4,5,133,4.422630,0.637348,Recital (86),"The controller should communicate to the data subject a personal data breach, wi"
4,5,8,368,3.826641,0.573150,Article 33 - Notification of a personal data breach to the supervisory authority,phases without undue further delay. 5. The controller shall document any persona
5,6,4,371,3.134781,0.659280,Article 34 - Communication of a personal data breach to the data subject,"would involve disproportionate effort. In such a case, there shall instead be a"
6,7,6,135,0.424893,0.593405,Recital (88),In setting detailed rules concerning the format and procedures applicable to the
7,8,7,369,-0.012659,0.591959,Article 34 - Communication of a personal data breach to the data subject,1. When the personal data breach is likely to result in a high risk to the right
8,9,9,367,-1.436821,0.573057,Article 33 - Notification of a personal data breach to the supervisory authority,"the nature of the personal data breach including where possible, the categories"
9,10,10,510,-3.644303,0.541692,Article 70 - Tasks of the Board,"issue guidelines, recommendations and best practices in accordance with point (e"


## 6. Answer generation with a shared prompt

Both pipelines go through the **same** `build_prompt()` function, so the only thing that differs between Answer 1
and Answer 2 is *which* 3 chunks are in the context (and their order).

The prompt tells the model to answer only from the supplied excerpts and to cite them. This matters with a famous
document like the GDPR: without it, the LLM could answer from its own memory and hide retrieval mistakes.

Gemini is called with `temperature=0` and thinking disabled to keep answers as deterministic as possible.
Responses are cached in `results/llm_cache.json` (keyed by a hash of the prompt) together with the latency that was
measured when they were generated. Re-running the notebook therefore reuses the same answers and does not need an API key;
set `USE_LLM_CACHE = False` to force fresh calls.

In [9]:
from google import genai
from google.genai import types

PROMPT_TEMPLATE = '''You are a legal assistant answering questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the numbered context excerpts below. Do not use outside knowledge.
- Cite the excerpts you rely on, e.g. [1], [2].
- State any conditions, exceptions or limitations that the excerpts attach to the rule.
- If the excerpts do not contain the answer, reply exactly: "Not found in the provided context."
Keep the answer under 120 words.

Context:
{context}

Question: {question}
Answer:'''


def build_prompt(question, selected):
    # Identical template for both pipelines; only the selected chunks differ.
    blocks = []
    for i, c in enumerate(selected, start=1):
        ch = chunks[c["chunk_id"]]
        blocks.append(f"[{i}] ({ch['section']}, page {ch['page']})\n{ch['text']}")
    return PROMPT_TEMPLATE.format(context="\n\n".join(blocks), question=question)


USE_LLM_CACHE = True
CACHE_PATH = RESULTS_DIR / "llm_cache.json"
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"]) if os.getenv("GEMINI_API_KEY") else None


def generate(prompt, retries=5):
    # Returns (answer_text, latency_seconds). Latency is the real API round-trip time.
    key = hashlib.sha256((GEMINI_MODEL + prompt).encode()).hexdigest()
    if USE_LLM_CACHE and key in llm_cache:
        hit = llm_cache[key]
        return hit["answer"], hit["latency_s"]
    if _client is None:
        raise RuntimeError("No cached answer and GEMINI_API_KEY is not set")
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            resp = _client.models.generate_content(
                model=GEMINI_MODEL, contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0, max_output_tokens=400,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)))
            latency = time.perf_counter() - t0
            break
        except Exception as e:                      # rate limit / transient error -> back off
            wait = 2 ** attempt * 5
            print(f"  Gemini error ({type(e).__name__}), retrying in {wait}s")
            time.sleep(wait)
    else:
        raise RuntimeError("Gemini call failed repeatedly")
    answer = resp.text.strip()
    llm_cache[key] = {"answer": answer, "latency_s": latency}
    CACHE_PATH.write_text(json.dumps(llm_cache, indent=1))
    return answer, latency


print("LLM:", GEMINI_MODEL, "| cached answers:", len(llm_cache))

LLM: gemini-2.5-flash | cached answers: 0


### 6.1 Running both pipelines for one query

`run_both()` executes the full flow and records timings for each stage:

`query → bi-encoder Top 10 → (P1) first 3 → LLM → Answer 1`
`                         ↘ (P2) cross-encoder re-sort → first 3 → LLM → Answer 2`

In [10]:
def run_both(question):
    t0 = time.perf_counter()
    top10 = retrieve_top_k(question)                 # shared stage 1
    t1 = time.perf_counter()
    reranked = rerank(question, top10)               # pipeline 2 only
    t2 = time.perf_counter()

    p1_ctx, p2_ctx = baseline_top_n(top10), reranked_top_n(reranked)
    answer1, llm1_s = generate(build_prompt(question, p1_ctx))
    answer2, llm2_s = generate(build_prompt(question, p2_ctx))

    return {"question": question, "top10": top10, "reranked": reranked,
            "p1_ids": [c["chunk_id"] for c in p1_ctx], "p2_ids": [c["chunk_id"] for c in p2_ctx],
            "answer1": answer1, "answer2": answer2,
            "timing": {"retrieval_ms": (t1 - t0) * 1e3, "rerank_ms": (t2 - t1) * 1e3,
                       "llm1_ms": llm1_s * 1e3, "llm2_ms": llm2_s * 1e3}}


def rank_table(res, gold_ids=()):
    # Top-10 table showing both rankings side by side; '★' marks gold chunks.
    df = pd.DataFrame(res["reranked"]).sort_values("bi_rank")
    df["gold"] = df.chunk_id.isin(gold_ids).map({True: "★", False: ""})
    df["section"] = df.chunk_id.map(chunks_df.section).str[:60]
    df["page"] = df.chunk_id.map(chunks_df.page)
    df["snippet"] = df.chunk_id.map(chunks_df.text).str[:70] + "…"
    df["in P1 top3"] = df.bi_rank <= TOP_N
    df["in P2 top3"] = df.ce_rank <= TOP_N
    return df[["gold", "chunk_id", "bi_rank", "bi_score", "ce_rank", "ce_score",
               "in P1 top3", "in P2 top3", "section", "page", "snippet"]].reset_index(drop=True)


def show_answers(res):
    display(Markdown(
        "| **Answer 1 – Baseline (bi-encoder Top 3)** | **Answer 2 – Re-ranked (cross-encoder Top 3)** |\n|---|---|\n"
        f"| {res['answer1'].replace(chr(10), '<br>')} | {res['answer2'].replace(chr(10), '<br>')} |\n"
        f"| context chunks: {res['p1_ids']} | context chunks: {res['p2_ids']} |"))


demo = run_both(demo_q)
display(rank_table(demo).style.format({"bi_score": "{:.3f}", "ce_score": "{:+.2f}"}))
show_answers(demo)

,gold,chunk_id,bi_rank,bi_score,ce_rank,ce_score,in P1 top3,in P2 top3,section,page,snippet
0,,366,1,0.777,1,+8.23,True,True,Article 33 - Notification of a personal data breach to the s,52,"1. In the case of a personal data breach, the controller shall without…"
1,,134,2,0.775,3,+6.31,True,True,Recital (87),17,It should be ascertained whether all appropriate technological protect…
2,,132,3,0.735,2,+7.90,True,True,Recital (85),16,"unauthorised reversal of pseudonymisation, damage to reputation, loss …"
3,,371,4,0.659,6,+3.13,False,False,Article 34 - Communication of a personal data breach to the,52,"would involve disproportionate effort. In such a case, there shall ins…"
4,,133,5,0.637,4,+4.42,False,False,Recital (86),17,The controller should communicate to the data subject a personal data …
5,,135,6,0.593,7,+0.42,False,False,Recital (88),17,In setting detailed rules concerning the format and procedures applica…
6,,369,7,0.592,8,-0.01,False,False,Article 34 - Communication of a personal data breach to the,52,1. When the personal data breach is likely to result in a high risk to…
7,,368,8,0.573,5,+3.83,False,False,Article 33 - Notification of a personal data breach to the s,52,phases without undue further delay. 5. The controller shall document a…
8,,367,9,0.573,9,-1.44,False,False,Article 33 - Notification of a personal data breach to the s,52,"the nature of the personal data breach including where possible, the c…"
9,,510,10,0.542,10,-3.64,False,False,Article 70 - Tasks of the Board,76,"issue guidelines, recommendations and best practices in accordance wit…"


| **Answer 1 – Baseline (bi-encoder Top 3)** | **Answer 2 – Re-ranked (cross-encoder Top 3)** |
|---|---|
| A personal data breach must be notified to the supervisory authority without undue delay and, where feasible, not later than 72 hours after the controller becomes aware of it [1], [3]. If the notification is not made within 72 hours, reasons for the delay must accompany it [1], [3]. This notification is not required if the breach is unlikely to result in a risk to the rights and freedoms of natural persons [1], [3]. | A personal data breach must be notified to the supervisory authority without undue delay and, where feasible, not later than 72 hours after the controller becomes aware of it [1], [2]. If the notification is not made within 72 hours, reasons for the delay must accompany it [1], [2]. This notification is not required if the personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons [1], [2]. |
| context chunks: [366, 134, 132] | context chunks: [366, 132, 134] |